In [3]:
from google.colab import drive
drive.mount('/content/drive')
path="/content/drive/MyDrive/DataMiningLab/ai_human_detection_v1.csv"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# Load and Clean
print("Loading and cleaning dataset...")
df = pd.read_csv(path)
df = df.dropna(subset=['text'])
df = df[~df['text'].str.contains("Error: 400|Error: 404", na=False)]
df['label'] = df['human_or_ai'].map({'human': 0, 'ai': 1})
df = df.dropna(subset=['label'])

X = df['text'].astype(str)
y = df['label'].astype(int)

# Split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Data ready. Total samples: {len(df)}")

Loading and cleaning dataset...
Data ready. Total samples: 514


In [5]:
print("Vectorizing text data...")
tfidf = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1, 2))
X_train = tfidf.fit_transform(X_train_raw)
X_test = tfidf.transform(X_test_raw)
print("Vectorization complete.")

Vectorizing text data...
Vectorization complete.


In [6]:
# Note: I'm using LogisticRegression instead of "Linear Regression"
# because Linear Regression is for numbers, while Logistic is for categories.

models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

results = []

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)

    results.append({"Model": name, "Accuracy": accuracy, "F1-Score": f1})
    print(f"{name} finished.")

# Display Comparison Table
comparison_df = pd.DataFrame(results)
print("\n--- Model Performance Comparison ---")
print(comparison_df.sort_values(by="Accuracy", ascending=False))

Training Random Forest...
Random Forest finished.
Training Logistic Regression...
Logistic Regression finished.
Training Naive Bayes...
Naive Bayes finished.
Training Decision Tree...
Decision Tree finished.

--- Model Performance Comparison ---
                 Model  Accuracy  F1-Score
3        Decision Tree  0.970874  0.976000
0        Random Forest  0.951456  0.959350
2          Naive Bayes  0.912621  0.928000
1  Logistic Regression  0.902913  0.926471
